# Exploring Time Series

## Plotting, resampling, and rolling averages

In the explainer, we saw how a single electricity dataset contains layers of pattern: a daily cycle, an annual season, a long-term trend. We also saw that these patterns are invisible when everything is tangled together in raw data. The job now is to untangle them.

In this notebook we play the role of a grid analyst examining a year of hourly energy data. Hourly readings are noisy — spikes, dips, and local variation make it hard to see what is actually happening. Our task is to find the signal beneath the noise using the pandas time series toolkit: datetime indexing, resampling, and rolling averages.

By the end we will be able to identify daily, weekly, and seasonal patterns in any time-indexed dataset.

In [ ]:
%pip install -q pandas matplotlib seaborn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)

---

## 1. Loading data and setting a DatetimeIndex

Before we can do any time-based analysis, pandas needs to know that our data is ordered in time. That means converting the timestamp column from a plain string into a **`datetime64`** object and promoting it to the index. This single step unlocks date-based filtering, resampling, and plotting.

In [ ]:
demand = pd.read_csv("../data/grid_demand_hourly.csv")
demand.head()

In [ ]:
demand.dtypes

The `timestamp` column loaded as a plain string (`object`). That is the default behaviour for CSV files. We need to convert it to a datetime and set it as the index so pandas treats this as a time series.

In [ ]:
demand["timestamp"] = pd.to_datetime(demand["timestamp"])
demand = demand.set_index("timestamp")
demand.index

The index is now a **`DatetimeIndex`**. This tells pandas the data is ordered in time, which means we can slice by date range, resample to different frequencies, and align multiple series by timestamp.

A shortcut for the two steps above: `pd.read_csv(path, parse_dates=["timestamp"], index_col="timestamp")`. We will use that shortcut when loading the solar and wind data later.

In [ ]:
demand.head()

---

## 2. Plotting raw hourly data

The first thing to do with any time series is plot it. We already know from the explainer that trend, seasonality, and noise are layered on top of each other in raw data. With 8,784 hourly points the plot will be dense, but that density is informative: it shows us just how much short-term variation sits on top of the longer-term patterns.

In [ ]:
demand["demand_mw"].plot(title="Hourly Grid Demand (2024)", ylabel="MW")
plt.tight_layout()
plt.show()

We should see a thick band of values with a clear seasonal shape: higher demand in winter (January and December), lower in summer. The daily cycle creates the vertical thickness of the band. There is too much noise to read the fine detail, which is exactly why we need resampling and smoothing. The goal for the rest of this notebook is to peel those layers apart.

---

## 3. Resampling: changing the frequency

The explainer described decomposition as the process of separating a time series into its components. Resampling is the first practical tool for doing that. **`.resample("D")`** groups all hours within each day and lets us apply an aggregation like `.mean()`. This collapses the within-day variation into a single value per day, revealing the day-to-day movement underneath.

In [ ]:
daily_demand = demand["demand_mw"].resample("D").mean()
daily_demand.head(10)

In [ ]:
daily_demand.plot(title="Daily Average Demand", ylabel="MW")
plt.tight_layout()
plt.show()

The seasonal pattern is much clearer now. We can also spot a regular weekly dip: weekends have lower demand than weekdays, because commercial and industrial consumption drops.

We can resample to weekly or monthly frequency to smooth even further.

In [ ]:
weekly_demand = demand["demand_mw"].resample("W").mean()
monthly_demand = demand["demand_mw"].resample("ME").mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

weekly_demand.plot(ax=axes[0], title="Weekly Average Demand", ylabel="MW")
monthly_demand.plot(ax=axes[1], title="Monthly Average Demand", ylabel="MW", marker="o")

plt.tight_layout()
plt.show()

The `"W"` code means week-ending Sunday. `"ME"` means month-end. Other useful codes: `"h"` (hourly), `"QE"` (quarter-end), `"YE"` (year-end). Each resampled point summarises all the original data points within that period.

Notice the trade-off: monthly resampling gives the cleanest seasonal shape, but we lose the week-to-week variation that might matter for short-term planning.

---

## 4. Rolling averages: smoothing without changing frequency

Resampling reduces the number of data points. Sometimes we want to smooth the noise without losing resolution. A **rolling average** (the moving average from the explainer) keeps every data point but replaces each value with the mean of its surrounding window. This is how we estimate the trend component the explainer described: average over a window long enough to wash out the short-term cycles.

In [ ]:
# 24-hour rolling average removes the daily cycle
demand["rolling_24h"] = demand["demand_mw"].rolling(window=24).mean()

# 168-hour (7-day) rolling average removes the weekly cycle
demand["rolling_7d"] = demand["demand_mw"].rolling(window=168).mean()

In [ ]:
# Plot two weeks to see the effect clearly
jan = demand.loc["2024-01-08":"2024-01-21"]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(jan.index, jan["demand_mw"], alpha=0.3, label="Hourly")
ax.plot(jan.index, jan["rolling_24h"], label="24h rolling avg", linewidth=2)
ax.plot(jan.index, jan["rolling_7d"], label="7d rolling avg", linewidth=2)
ax.set_title("Demand: Raw vs Rolling Averages (January 2024)")
ax.set_ylabel("MW")
ax.legend()
plt.tight_layout()
plt.show()

The 24-hour rolling average filters out the daily peak-and-trough cycle, leaving the day-to-day movement. The 7-day rolling average goes further and removes the weekly pattern too, showing only the longer-term trend. This is decomposition in action: each wider window strips away another layer of variation.

The first `window - 1` values will be `NaN` because there are not enough preceding points to fill the window. That is expected.

---

## 5. Full-year rolling average

Let's apply the 7-day rolling average across the entire year. This should reveal the seasonal trend the explainer described: the long-term direction that would still be there if we removed all the short-term fluctuations.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(demand.index, demand["demand_mw"], alpha=0.15, label="Hourly", color="grey")
ax.plot(demand.index, demand["rolling_7d"], label="7-day rolling avg", color="#d62728", linewidth=2)
ax.set_title("Grid Demand with 7-Day Rolling Average (2024)")
ax.set_ylabel("MW")
ax.legend()
plt.tight_layout()
plt.show()

The red line shows the underlying seasonal pattern: demand peaks in January, drops through spring and summer, then climbs again in autumn. This is the heating-driven annual seasonality we read about in the explainer. The grey cloud of hourly data around it is mostly daily cycles and noise.

---

## 6. Comparing demand with solar and wind

So far we have been looking at demand in isolation. But grid operators care about the relationship between demand and supply. The explainer mentioned that wind now accounts for around 30% of UK generation. The question for the grid is whether renewable supply lines up with demand, or whether they move in opposite directions. Let's load the solar and wind data and find out.

In [ ]:
solar = pd.read_csv("../data/solar_output.csv", parse_dates=["timestamp"], index_col="timestamp")
wind = pd.read_csv("../data/wind_output.csv", parse_dates=["timestamp"], index_col="timestamp")

print(f"Solar: {solar.shape}")
print(f"Wind:  {wind.shape}")

In [ ]:
# Daily averages for all three
daily = pd.DataFrame({
    "demand": demand["demand_mw"].resample("D").mean(),
    "solar": solar["output_mw"].resample("D").mean(),
    "wind": wind["output_mw"].resample("D").mean(),
})
daily.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(daily.index, daily["demand"], label="Demand", linewidth=1.5)
ax.plot(daily.index, daily["solar"], label="Solar", linewidth=1.5)
ax.plot(daily.index, daily["wind"], label="Wind", linewidth=1.5)
ax.set_title("Daily Average: Demand vs Solar vs Wind (2024)")
ax.set_ylabel("MW")
ax.legend()
plt.tight_layout()
plt.show()

Notice the inverse relationship: **demand is highest in winter, but solar output is highest in summer**. Wind partially compensates, since it tends to be stronger in winter, but the combined renewable output still falls short of demand in the coldest months. This seasonal mismatch is the central challenge for grid planning, and it is the problem the next two notebooks will try to address with forecasting and scenario analysis.

In [ ]:
# Monthly averages to see the seasonal shape even more clearly
monthly = daily.resample("ME").mean()

monthly.plot(kind="bar", figsize=(14, 5), title="Monthly Average Output (MW)")
plt.ylabel("MW")
plt.xticks(range(12), ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                        "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], rotation=0)
plt.tight_layout()
plt.show()

---

## 7. Identifying daily patterns

Resampling and rolling averages reveal patterns across days and months. But the explainer also described the daily cycle: the 2 AM trough and 6 PM peak that repeats every 24 hours. We can isolate that pattern by grouping by hour of day.

In [ ]:
hourly_profile = demand["demand_mw"].groupby(demand.index.hour).mean()

fig, ax = plt.subplots(figsize=(10, 5))
hourly_profile.plot(ax=ax, marker="o")
ax.set_title("Average Demand by Hour of Day")
ax.set_xlabel("Hour")
ax.set_ylabel("MW")
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

Demand is lowest between midnight and 05:00, rises sharply in the morning as the country wakes up, holds through the afternoon, peaks around 18:00, then declines through the evening. This is the daily seasonality the explainer described, and it repeats with remarkable consistency.

In [ ]:
# Solar profile by hour
solar_hourly = solar["output_mw"].groupby(solar.index.hour).mean()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hourly_profile.index, hourly_profile.values, marker="o", label="Demand")
ax.plot(solar_hourly.index, solar_hourly.values, marker="s", label="Solar")
ax.set_title("Average Hourly Profile: Demand vs Solar")
ax.set_xlabel("Hour")
ax.set_ylabel("MW")
ax.set_xticks(range(0, 24))
ax.legend()
plt.tight_layout()
plt.show()

Solar peaks at midday, but demand peaks at 18:00. By the time people get home and turn on lights and cookers, the sun is going down. This timing mismatch is a second planning headache on top of the seasonal mismatch we saw earlier. Both problems will matter when we start forecasting the supply gap.

---

## Exercises

We have seen how resampling, rolling averages, and time-component grouping reveal the structure hidden inside raw time series. These exercises give you a chance to apply those techniques independently.

### Exercise 1: Weekly demand pattern

Group the demand data by **day of week** (use `demand.index.dayofweek`, where 0 = Monday, 6 = Sunday) and calculate the mean demand for each day. Plot the result as a bar chart. Which day has the highest average demand? Which has the lowest?

In [ ]:
# Your code here


### Exercise 2: Summer vs winter solar

Filter the solar data to January only and to July only. For each month, compute the average hourly output profile (group by hour of day). Plot both profiles on the same chart. How much more solar output does a typical July hour produce compared to a typical January hour at midday?

In [ ]:
# Your code here


### Exercise 3: Wind rolling average

Load the wind output data (if not already loaded). Calculate a 72-hour (3-day) rolling average of `output_mw`. Plot the raw hourly wind output and the rolling average together for the month of March 2024. What do the multi-day high and low periods represent?

In [ ]:
# Your code here


### Exercise 4: Total renewable output

Create a new column in the `daily` DataFrame called `"total_renewable"` that is the sum of `solar` and `wind`. Then calculate `"supply_gap"` as `demand - total_renewable`. Plot the supply gap over the full year with a 7-day rolling average. In which months is the supply gap largest?

In [ ]:
# Your code here


---

## Summary

This notebook covered the core techniques for exploring time series data:

- **`pd.to_datetime`** and **`set_index`** convert a string column into a `DatetimeIndex`, which unlocks all of pandas' time series functionality.
- **`.resample()`** aggregates data to a lower frequency (daily, weekly, monthly), stripping away higher-frequency variation to reveal longer-term patterns.
- **`.rolling()`** applies a sliding window to smooth data without changing the frequency. A 24-hour window removes the daily cycle; a 168-hour window removes the weekly cycle.
- **Grouping by time components** (`.index.hour`, `.index.dayofweek`, `.index.month`) isolates the repeating patterns at each scale — the seasonal components the explainer described.
- Plotting demand alongside solar and wind output exposes the fundamental mismatches that grid planners must solve: seasonal (winter demand vs summer solar) and daily (evening peak vs midday solar).

In the next notebook, we will use these same tools to build actual forecasts and measure how accurate they are.